# 07 Final Reviewer Runs

Put this notebook in the same project folder as notebooks 1--6 and run it from top to bottom. It does three things:

1. Re-runs notebooks 1--6 if enabled.
2. Collects prediction/evaluation outputs and recomputes reviewer-facing numbers: RMSE tables, paired bootstrap CIs, seed robustness summaries, and diagnostic plots.
3. Writes final paper-ready outputs to `final_reviewer_outputs/`.

The first code cell is the only place you should usually edit paths/flags.


In [ ]:
# ============================================================
# CONFIG — edit this cell only if your filenames differ
# ============================================================
from pathlib import Path
import os, sys, re, json, math, glob, shutil, subprocess, warnings, ast
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
OUT = ROOT / "final_reviewer_outputs"
FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables"
LOG_DIR = OUT / "logs"
EXEC_DIR = OUT / "executed_notebooks"
for p in [OUT, FIG_DIR, TABLE_DIR, LOG_DIR, EXEC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Run notebooks 1--6 first. Set False if you already ran them manually.
RUN_NOTEBOOKS_1_TO_6 = True

# Auto-detect notebooks beginning with 1,2,3,4,5,6.
# If your names are weird, directly list them here instead, e.g. ["01_data.ipynb", ...]
NOTEBOOK_FILES = []
NOTEBOOK_PATTERNS = ["1*.ipynb", "01*.ipynb", "2*.ipynb", "02*.ipynb", "3*.ipynb", "03*.ipynb", "4*.ipynb", "04*.ipynb", "5*.ipynb", "05*.ipynb", "6*.ipynb", "06*.ipynb"]

# Prediction/evaluation CSV search paths.
# The notebook will auto-search these. Add exact paths if needed.
PREDICTION_CSV_GLOBS = [
    "results/**/*.csv", "outputs/**/*.csv", "figures/**/*.csv", "tables/**/*.csv",
    "eval/**/*.csv", "evaluation/**/*.csv", "runs/**/*.csv", "notebook_outputs/**/*.csv",
    "*.csv"
]

# If you have a single known per-window prediction CSV, put it here.
# Required columns can be flexible, but ideally: dataset, method, task, horizon, window_id, y_true, y_pred.
FORCE_PREDICTION_FILES = []

# Bootstrap settings. 5000 is paper-ish; use 1000 for a fast sanity run.
BOOTSTRAP_B = 5000
BOOTSTRAP_SEED = 123

# Method labels used in paper. The cleaner below will map common variants to these.
PAPER_METHOD_ORDER = ["LOCF", "LinearInterp + SeasonalNaive", "MAR (LDS)", "MNAR (LDS)"]
PAPER_DATASET_ORDER = ["Seattle", "METR-LA", "Synthetic"]

# Reviewer-requested horizon labels.
HORIZON_ORDER = ["impute", "1-step", "3-step", "6-step"]

# Length buckets. Empty buckets are automatically dropped in plots/tables.
LENGTH_BINS = [0, 6, 12, 24, 72, np.inf]
LENGTH_LABELS = ["1-6", "7-12", "13-24", "25-72", "73+"]

print("ROOT:", ROOT)
print("Outputs will be written to:", OUT)


In [ ]:
# ============================================================
# 1) Run notebooks 1--6
# ============================================================
def find_base_notebooks():
    if NOTEBOOK_FILES:
        files = [ROOT / f for f in NOTEBOOK_FILES]
    else:
        found = []
        for pat in NOTEBOOK_PATTERNS:
            found.extend(ROOT.glob(pat))
        # unique + exclude this final notebook and executed notebooks
        files = []
        seen = set()
        for f in found:
            if not f.is_file():
                continue
            name = f.name.lower()
            if "final_reviewer" in name or "executed" in str(f).lower() or ".ipynb_checkpoints" in str(f):
                continue
            if f.resolve() not in seen:
                seen.add(f.resolve())
                files.append(f)
        files = sorted(files, key=lambda p: p.name)
    return files

base_notebooks = find_base_notebooks()
print("Detected base notebooks:")
for nb in base_notebooks:
    print(" -", nb.name)

if RUN_NOTEBOOKS_1_TO_6:
    if not base_notebooks:
        print("WARNING: No notebooks detected. Check NOTEBOOK_FILES in config.")
    for nb in base_notebooks:
        out_name = nb.stem + "__executed.ipynb"
        log_path = LOG_DIR / f"{nb.stem}.log"
        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute", str(nb),
            "--ExecutePreprocessor.timeout=-1",
            f"--ExecutePreprocessor.cwd={ROOT}",
            "--output", out_name,
            "--output-dir", str(EXEC_DIR),
        ]
        print("\nRunning:", nb.name)
        with open(log_path, "w", encoding="utf-8") as logf:
            proc = subprocess.run(cmd, cwd=ROOT, stdout=logf, stderr=subprocess.STDOUT, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"Notebook failed: {nb.name}. See log: {log_path}")
        print("Done:", nb.name, "| log:", log_path)
else:
    print("Skipping notebook execution because RUN_NOTEBOOKS_1_TO_6=False")


In [ ]:
# ============================================================
# 2) Utility functions: parsing, standardization, RMSE, bootstrap
# ============================================================
def clean_method(x):
    if pd.isna(x): return x
    s = str(x).strip()
    low = s.lower().replace("_", " ").replace("-", " ")
    if low in {"locf", "last observation carried forward", "last carried forward"} or "locf" in low:
        return "LOCF"
    if "linear" in low and ("season" in low or "naive" in low):
        return "LinearInterp + SeasonalNaive"
    if "mnar" in low:
        return "MNAR (LDS)"
    if "mar" in low and "mnar" not in low:
        return "MAR (LDS)"
    if low in {"lds", "kalman", "kf"}:
        return "MAR (LDS)"
    return s

def clean_dataset(x, source=""):
    text = (str(x) if not pd.isna(x) else "") + " " + str(source)
    low = text.lower()
    if "metr" in low or "la" in low and "metr" in low:
        return "METR-LA"
    if "seattle" in low or "loop" in low:
        return "Seattle"
    if "synthetic" in low or "alpha" in low:
        return "Synthetic"
    return str(x).strip() if not pd.isna(x) and str(x).strip() else "Unknown"

def clean_task_horizon(task=None, horizon=None, source=""):
    text = " ".join([str(task) if task is not None else "", str(horizon) if horizon is not None else "", str(source)])
    low = text.lower()
    if "imput" in low or "recon" in low:
        return "impute"
    # detect horizons
    for h in [1,3,6]:
        patterns = [f"{h}-step", f"{h} step", f"h{h}", f"h={h}", f"horizon_{h}", f"horizon {h}", f"_{h}step", f"{h}step"]
        if any(p in low for p in patterns):
            return f"{h}-step"
    # numeric horizon value
    try:
        hv = int(float(str(horizon)))
        if hv in [1,3,6]: return f"{hv}-step"
    except Exception:
        pass
    if "forecast" in low or "pred" in low:
        return "forecast"
    return str(task).strip() if task is not None and not pd.isna(task) and str(task).strip() else "unknown"

def first_col(df, candidates):
    lower_to_col = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns: return c
        if c.lower() in lower_to_col: return lower_to_col[c.lower()]
    # fuzzy contains
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def parse_numeric_array(v):
    """Return np.array of floats from scalar/list/string. Empty array if impossible."""
    if isinstance(v, (list, tuple, np.ndarray)):
        arr = np.asarray(v, dtype=float).ravel()
        return arr[np.isfinite(arr)]
    if pd.isna(v):
        return np.array([], dtype=float)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return np.array([float(v)], dtype=float)
    s = str(v).strip()
    if not s:
        return np.array([], dtype=float)
    # Try JSON / Python literal first
    for loader in (json.loads, ast.literal_eval):
        try:
            obj = loader(s)
            arr = np.asarray(obj, dtype=float).ravel()
            return arr[np.isfinite(arr)]
        except Exception:
            pass
    # Fallback: extract all numbers from string
    nums = re.findall(r"[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+(?:[eE][-+]?\d+)?", s)
    if nums:
        arr = np.asarray([float(n) for n in nums], dtype=float)
        return arr[np.isfinite(arr)]
    return np.array([], dtype=float)

def row_sqerr_sum_n(y_true, y_pred):
    yt = parse_numeric_array(y_true)
    yp = parse_numeric_array(y_pred)
    n = min(len(yt), len(yp))
    if n == 0:
        return np.nan, 0
    diff = yp[:n] - yt[:n]
    return float(np.sum(diff * diff)), int(n)

def rmse_from_sqerr(df):
    n = df["n_obs"].sum()
    if n <= 0:
        return np.nan
    return float(np.sqrt(df["sqerr_sum"].sum() / n))

def bootstrap_rmse_ci(df, B=BOOTSTRAP_B, seed=BOOTSTRAP_SEED, id_col="window_id"):
    rng = np.random.default_rng(seed)
    if id_col not in df.columns:
        # fallback row bootstrap
        ids = np.arange(len(df))
        tmp = df.copy()
        tmp[id_col] = ids
    else:
        tmp = df.copy()
        ids = tmp[id_col].dropna().unique()
    if len(ids) == 0:
        return np.nan, np.nan, np.nan
    vals = []
    grouped = {k: g for k, g in tmp.groupby(id_col)}
    for _ in range(B):
        sample_ids = rng.choice(ids, size=len(ids), replace=True)
        sq = 0.0; n = 0
        for sid in sample_ids:
            g = grouped[sid]
            sq += g["sqerr_sum"].sum()
            n += g["n_obs"].sum()
        vals.append(np.sqrt(sq / n) if n > 0 else np.nan)
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    point = rmse_from_sqerr(tmp)
    return point, float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def paired_delta_bootstrap(df, method_a="MNAR (LDS)", method_b="MAR (LDS)", B=BOOTSTRAP_B, seed=BOOTSTRAP_SEED):
    # delta = RMSE(method_a) - RMSE(method_b), negative favors method_a if method_a=MNAR
    d = df[df["method"].isin([method_a, method_b])].copy()
    if d.empty or "window_id" not in d.columns:
        return None
    # per method/window aggregate sqerr/n
    agg = d.groupby(["method", "window_id"], as_index=False).agg(sqerr_sum=("sqerr_sum", "sum"), n_obs=("n_obs", "sum"))
    pivot_sq = agg.pivot(index="window_id", columns="method", values="sqerr_sum")
    pivot_n = agg.pivot(index="window_id", columns="method", values="n_obs")
    common = pivot_sq.dropna(subset=[method_a, method_b]).index.intersection(pivot_n.dropna(subset=[method_a, method_b]).index)
    if len(common) == 0:
        return None
    sq_a = pivot_sq.loc[common, method_a].to_numpy(dtype=float)
    sq_b = pivot_sq.loc[common, method_b].to_numpy(dtype=float)
    n_a = pivot_n.loc[common, method_a].to_numpy(dtype=float)
    n_b = pivot_n.loc[common, method_b].to_numpy(dtype=float)
    def delta(indices):
        ra = np.sqrt(sq_a[indices].sum() / n_a[indices].sum())
        rb = np.sqrt(sq_b[indices].sum() / n_b[indices].sum())
        return ra - rb
    idx = np.arange(len(common))
    point = delta(idx)
    rng = np.random.default_rng(seed)
    vals = np.array([delta(rng.choice(idx, size=len(idx), replace=True)) for _ in range(B)], dtype=float)
    return {
        "n_windows": int(len(common)),
        "delta_rmse": float(point),
        "ci_low": float(np.percentile(vals, 2.5)),
        "ci_high": float(np.percentile(vals, 97.5)),
    }

print("Utilities loaded.")


In [ ]:
# ============================================================
# 3) Load and standardize prediction/evaluation CSVs
# ============================================================
def candidate_csv_files():
    files = []
    if FORCE_PREDICTION_FILES:
        files.extend([ROOT / f for f in FORCE_PREDICTION_FILES])
    for pat in PREDICTION_CSV_GLOBS:
        files.extend(ROOT.glob(pat))
    # exclude our final output files to avoid self-ingestion
    clean = []
    seen = set()
    for f in files:
        if not f.is_file():
            continue
        if OUT in f.parents:
            continue
        if f.resolve() in seen:
            continue
        seen.add(f.resolve())
        clean.append(f)
    return sorted(clean)

def standardize_prediction_csv(path):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if df.empty:
        return None
    cols = set(c.lower() for c in df.columns)
    # Skip obvious aggregate-only files unless they have enough prediction columns.
    true_col = first_col(df, ["y_true", "true", "actual", "target", "gt", "ground_truth", "x_true"])
    pred_col = first_col(df, ["y_pred", "pred", "prediction", "forecast", "imputed", "x_pred", "x_hat"])
    err_col = first_col(df, ["sqerr_sum", "squared_error_sum", "se_sum"])
    rmse_col = first_col(df, ["rmse"])
    method_col = first_col(df, ["method", "model", "model_name", "approach"])
    if method_col is None:
        # infer from file name if possible, but still need predictions/errors
        if not any(k in path.name.lower() for k in ["locf", "mar", "mnar", "linear", "season"]):
            return None
    if true_col is None and pred_col is None and err_col is None and rmse_col is None:
        return None

    out = pd.DataFrame()
    out["source_file"] = str(path.relative_to(ROOT))
    out["method"] = df[method_col].map(clean_method) if method_col else clean_method(path.stem)

    dataset_col = first_col(df, ["dataset", "data", "city"])
    out["dataset"] = df[dataset_col].map(lambda x: clean_dataset(x, path)) if dataset_col else clean_dataset(None, path)

    task_col = first_col(df, ["task", "eval_task", "split_task", "type"])
    horizon_col = first_col(df, ["horizon", "h", "step", "forecast_horizon"])
    task_vals = df[task_col] if task_col else pd.Series([None]*len(df))
    hor_vals = df[horizon_col] if horizon_col else pd.Series([None]*len(df))
    out["task_horizon"] = [clean_task_horizon(t, h, path) for t, h in zip(task_vals, hor_vals)]

    win_col = first_col(df, ["window_id", "window", "event_id", "blackout_id", "id"])
    if win_col:
        out["window_id"] = df[win_col].astype(str)
    else:
        # stable fallback: one row is one pseudo-window
        out["window_id"] = [f"{path.stem}_row{i}" for i in range(len(df))]

    det_col = first_col(df, ["detector", "detector_id", "det", "sensor", "sensor_id"])
    if det_col: out["detector"] = df[det_col].astype(str)

    len_col = first_col(df, ["blackout_len", "length", "len", "duration", "blackout_length"])
    if len_col:
        out["blackout_len"] = pd.to_numeric(df[len_col], errors="coerce")
    else:
        out["blackout_len"] = np.nan

    time_col = first_col(df, ["start_time", "timestamp", "time", "start", "blackout_start"])
    if time_col:
        out["start_time"] = pd.to_datetime(df[time_col], errors="coerce")
    else:
        out["start_time"] = pd.NaT

    seed_col = first_col(df, ["seed", "random_seed", "run_seed"])
    if seed_col:
        out["seed"] = pd.to_numeric(df[seed_col], errors="coerce")
    else:
        out["seed"] = np.nan

    alpha_col = first_col(df, ["alpha", "mnar_alpha", "strength"])
    if alpha_col:
        out["alpha"] = pd.to_numeric(df[alpha_col], errors="coerce")
    else:
        out["alpha"] = np.nan

    if err_col and first_col(df, ["n_obs", "n", "count", "num_obs"]):
        n_col = first_col(df, ["n_obs", "n", "count", "num_obs"])
        out["sqerr_sum"] = pd.to_numeric(df[err_col], errors="coerce")
        out["n_obs"] = pd.to_numeric(df[n_col], errors="coerce").fillna(1).astype(float)
    elif true_col and pred_col:
        vals = [row_sqerr_sum_n(t, p) for t, p in zip(df[true_col], df[pred_col])]
        out["sqerr_sum"] = [v[0] for v in vals]
        out["n_obs"] = [v[1] for v in vals]
    elif rmse_col:
        # Aggregate-only fallback. Treat each row as one pseudo-observation with squared error = RMSE^2.
        # Useful for table creation, NOT valid for paired bootstrap.
        r = pd.to_numeric(df[rmse_col], errors="coerce")
        out["sqerr_sum"] = r ** 2
        out["n_obs"] = 1.0
        out["aggregate_only"] = True
    else:
        return None

    out["aggregate_only"] = out.get("aggregate_only", False)
    out = out[(out["n_obs"] > 0) & np.isfinite(out["sqerr_sum"])]
    if out.empty:
        return None
    return out

csvs = candidate_csv_files()
print(f"Found {len(csvs)} CSV candidates.")
frames = []
for f in csvs:
    sdf = standardize_prediction_csv(f)
    if sdf is not None and not sdf.empty:
        frames.append(sdf)
        print("Using:", f.relative_to(ROOT), "rows=", len(sdf))

if not frames:
    raise RuntimeError(
        "No usable prediction/evaluation CSVs found. Either run notebooks 1-6 first, "
        "or add exact files to FORCE_PREDICTION_FILES in the config cell. "
        "Best format: dataset, method, task, horizon, window_id, y_true, y_pred."
    )

pred = pd.concat(frames, ignore_index=True)
pred["method"] = pred["method"].map(clean_method)
pred["dataset"] = [clean_dataset(d, s) for d, s in zip(pred["dataset"], pred["source_file"])]
pred["task_horizon"] = [clean_task_horizon(th, None, s) for th, s in zip(pred["task_horizon"], pred["source_file"])]

# Drop unknown/garbage rows cautiously.
pred = pred[pred["method"].notna()]
pred.to_csv(TABLE_DIR / "standardized_predictions_all.csv", index=False)
print("\nStandardized prediction rows:", len(pred))
print(pred[["dataset", "method", "task_horizon", "window_id", "sqerr_sum", "n_obs", "source_file"]].head())
print("Saved:", TABLE_DIR / "standardized_predictions_all.csv")


In [ ]:
# ============================================================
# 4) Main RMSE tables + paired bootstrap uncertainty
# ============================================================
# Main RMSE by dataset/method/task-horizon
main_rows = []
for (dataset, task_horizon, method), g in pred.groupby(["dataset", "task_horizon", "method"]):
    main_rows.append({
        "dataset": dataset,
        "task_horizon": task_horizon,
        "method": method,
        "rmse": rmse_from_sqerr(g),
        "n_rows": len(g),
        "n_obs": int(g["n_obs"].sum()),
        "n_windows": g["window_id"].nunique() if "window_id" in g.columns else np.nan,
        "aggregate_only": bool(g.get("aggregate_only", pd.Series([False])).any()),
    })
main = pd.DataFrame(main_rows)
main["method_order"] = main["method"].map({m:i for i,m in enumerate(PAPER_METHOD_ORDER)}).fillna(999)
main["horizon_order"] = main["task_horizon"].map({h:i for i,h in enumerate(HORIZON_ORDER)}).fillna(999)
main = main.sort_values(["dataset", "horizon_order", "method_order", "method"])
main.to_csv(TABLE_DIR / "main_rmse_long.csv", index=False)
print("Saved:", TABLE_DIR / "main_rmse_long.csv")

for dataset in main["dataset"].unique():
    print("\n===", dataset, "===")
    piv = main[main["dataset"]==dataset].pivot_table(index="method", columns="task_horizon", values="rmse", aggfunc="first")
    cols = [c for c in HORIZON_ORDER if c in piv.columns] + [c for c in piv.columns if c not in HORIZON_ORDER]
    idx = [m for m in PAPER_METHOD_ORDER if m in piv.index] + [m for m in piv.index if m not in PAPER_METHOD_ORDER]
    print(piv.loc[idx, cols].round(3))
    piv.loc[idx, cols].to_csv(TABLE_DIR / f"{dataset.lower().replace('-', '').replace(' ', '_')}_rmse_wide.csv")

# Paired MNAR-MAR bootstrap deltas
boot_rows = []
for (dataset, task_horizon), g in pred.groupby(["dataset", "task_horizon"]):
    res = paired_delta_bootstrap(g, "MNAR (LDS)", "MAR (LDS)")
    if res is not None:
        boot_rows.append({"dataset": dataset, "task_horizon": task_horizon, **res})
boot = pd.DataFrame(boot_rows)
if not boot.empty:
    boot["horizon_order"] = boot["task_horizon"].map({h:i for i,h in enumerate(HORIZON_ORDER)}).fillna(999)
    boot = boot.sort_values(["dataset", "horizon_order"])
    boot.to_csv(TABLE_DIR / "paired_bootstrap_mnar_minus_mar.csv", index=False)
    print("\nPaired bootstrap MNAR-MAR deltas saved:", TABLE_DIR / "paired_bootstrap_mnar_minus_mar.csv")
    print(boot.drop(columns=["horizon_order"]).round(4))
else:
    print("\nWARNING: Could not compute paired bootstrap deltas. Need per-window predictions for both MAR and MNAR with matching window_id.")


In [ ]:
# ============================================================
# 5) Seed-level robustness / std summaries
# ============================================================
seed_df = pred[pred["seed"].notna()].copy()
seed_rows = []
if not seed_df.empty:
    for (dataset, task_horizon, method, seed), g in seed_df.groupby(["dataset", "task_horizon", "method", "seed"]):
        seed_rows.append({
            "dataset": dataset, "task_horizon": task_horizon, "method": method, "seed": int(seed),
            "rmse": rmse_from_sqerr(g), "n_windows": g["window_id"].nunique()
        })
    seed_long = pd.DataFrame(seed_rows)
    seed_long.to_csv(TABLE_DIR / "seed_level_rmse_long.csv", index=False)
    seed_summary = seed_long.groupby(["dataset", "task_horizon", "method"], as_index=False).agg(
        mean_rmse=("rmse", "mean"), std_rmse=("rmse", "std"), n_seeds=("seed", "nunique")
    )
    seed_summary.to_csv(TABLE_DIR / "seed_robustness_summary.csv", index=False)
    print("Saved:", TABLE_DIR / "seed_level_rmse_long.csv")
    print("Saved:", TABLE_DIR / "seed_robustness_summary.csv")
    print(seed_summary.round(4))

    # MNAR-MAR seed deltas where possible
    delta_rows = []
    piv = seed_long.pivot_table(index=["dataset", "task_horizon", "seed"], columns="method", values="rmse", aggfunc="first").reset_index()
    if "MNAR (LDS)" in piv.columns and "MAR (LDS)" in piv.columns:
        piv["delta_rmse"] = piv["MNAR (LDS)"] - piv["MAR (LDS)"]
        delta_summary = piv.groupby(["dataset", "task_horizon"], as_index=False).agg(
            mean_delta=("delta_rmse", "mean"), std_delta=("delta_rmse", "std"), n_seeds=("seed", "nunique")
        )
        delta_summary.to_csv(TABLE_DIR / "seed_mnar_minus_mar_deltas.csv", index=False)
        print("Saved:", TABLE_DIR / "seed_mnar_minus_mar_deltas.csv")
        print(delta_summary.round(4))
else:
    print("No seed column found in standardized outputs.")
    print("Reviewer-safe wording if needed: report bootstrap CIs over windows, and only claim seed robustness if you actually reran 5 seeds.")


In [ ]:
# ============================================================
# 6) Paper figures with clearer labels and empty-bucket handling
# ============================================================
def method_order_list(methods):
    return [m for m in PAPER_METHOD_ORDER if m in methods] + [m for m in sorted(methods) if m not in PAPER_METHOD_ORDER]

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

# 6.1 Seattle imputation RMSE with bootstrap CIs by method
sea_imp = pred[(pred["dataset"]=="Seattle") & (pred["task_horizon"]=="impute")].copy()
if not sea_imp.empty:
    rows = []
    for method, g in sea_imp.groupby("method"):
        point, lo, hi = bootstrap_rmse_ci(g)
        rows.append({"method": method, "rmse": point, "ci_low": lo, "ci_high": hi})
    ci_methods = pd.DataFrame(rows)
    ci_methods["order"] = ci_methods["method"].map({m:i for i,m in enumerate(PAPER_METHOD_ORDER)}).fillna(999)
    ci_methods = ci_methods.sort_values(["order", "method"])
    ci_methods.to_csv(TABLE_DIR / "seattle_impute_method_bootstrap_ci.csv", index=False)

    x = np.arange(len(ci_methods))
    y = ci_methods["rmse"].to_numpy()
    yerr = np.vstack([y - ci_methods["ci_low"].to_numpy(), ci_methods["ci_high"].to_numpy() - y])
    plt.figure(figsize=(7.2, 4.2))
    plt.bar(x, y)
    plt.errorbar(x, y, yerr=yerr, fmt="none", capsize=4)
    plt.xticks(x, ci_methods["method"], rotation=20, ha="right")
    plt.ylabel("RMSE (mph)")
    plt.title("Seattle Loop imputation RMSE with 95% bootstrap CIs")
    savefig(FIG_DIR / "impute_rmse_methods_ci_labels.png")
else:
    print("No Seattle imputation rows found; skipping imputation CI plot.")

# 6.2 Forecast RMSE vs horizon with CIs
sea_fore = pred[(pred["dataset"]=="Seattle") & (pred["task_horizon"].isin(["1-step", "3-step", "6-step"]))].copy()
if not sea_fore.empty:
    rows = []
    for (method, task_horizon), g in sea_fore.groupby(["method", "task_horizon"]):
        point, lo, hi = bootstrap_rmse_ci(g)
        rows.append({"method": method, "task_horizon": task_horizon, "rmse": point, "ci_low": lo, "ci_high": hi})
    fc = pd.DataFrame(rows)
    fc["h"] = fc["task_horizon"].str.extract(r"(\d+)").astype(float)
    fc.to_csv(TABLE_DIR / "seattle_forecast_method_bootstrap_ci.csv", index=False)

    plt.figure(figsize=(7.2, 4.2))
    for method in method_order_list(fc["method"].unique()):
        gm = fc[fc["method"]==method].sort_values("h")
        if gm.empty: continue
        x = gm["h"].to_numpy()
        y = gm["rmse"].to_numpy()
        yerr = np.vstack([y - gm["ci_low"].to_numpy(), gm["ci_high"].to_numpy() - y])
        plt.errorbar(x, y, yerr=yerr, marker="o", capsize=4, label=method)
    plt.xticks([1,3,6], ["1-step\n5 min", "3-step\n15 min", "6-step\n30 min"])
    plt.ylabel("RMSE (mph)")
    plt.xlabel("Forecast horizon")
    plt.title("Seattle Loop post-blackout forecasting RMSE")
    plt.legend(frameon=False)
    savefig(FIG_DIR / "forecast_rmse_by_horizon_panels_ci.png")
else:
    print("No Seattle forecast rows found; skipping forecast CI plot.")

# 6.3 Imputation RMSE by blackout length bucket; drop empty buckets automatically
if not sea_imp.empty and sea_imp["blackout_len"].notna().any():
    d = sea_imp.copy()
    d["length_bucket"] = pd.cut(d["blackout_len"], bins=LENGTH_BINS, labels=LENGTH_LABELS, include_lowest=True, right=True)
    bucket_counts = d.groupby("length_bucket", observed=True)["window_id"].nunique().rename("n_windows").reset_index()
    rows = []
    for (method, bucket), g in d.groupby(["method", "length_bucket"], observed=True):
        rows.append({"method": method, "length_bucket": str(bucket), "rmse": rmse_from_sqerr(g), "n_windows": g["window_id"].nunique()})
    bylen = pd.DataFrame(rows)
    bylen.to_csv(TABLE_DIR / "seattle_impute_rmse_by_length_bucket.csv", index=False)

    buckets = [b for b in LENGTH_LABELS if b in set(bylen["length_bucket"])]
    x = np.arange(len(buckets))
    plt.figure(figsize=(7.2, 4.2))
    for method in method_order_list(bylen["method"].unique()):
        gm = bylen[bylen["method"]==method].set_index("length_bucket").reindex(buckets)
        plt.plot(x, gm["rmse"], marker="o", label=method)
    count_map = bucket_counts.set_index("length_bucket")["n_windows"].to_dict()
    labels = [f"{b}\n(n={int(count_map.get(b, 0))})" for b in buckets]
    plt.xticks(x, labels)
    plt.xlabel("Blackout length bucket (5-min steps)")
    plt.ylabel("RMSE (mph)")
    plt.title("Seattle Loop imputation RMSE by blackout length")
    plt.legend(frameon=False)
    savefig(FIG_DIR / "impute_rmse_by_length_bucket_counts.png")
else:
    print("No blackout_len found for Seattle imputation; skipping length-bucket plot.")

# 6.4 Heatmaps by length/hour for MAR and MNAR
if not sea_imp.empty and sea_imp["blackout_len"].notna().any() and sea_imp["start_time"].notna().any():
    d = sea_imp.copy()
    d["length_bucket"] = pd.cut(d["blackout_len"], bins=LENGTH_BINS, labels=LENGTH_LABELS, include_lowest=True, right=True)
    d["hour"] = pd.to_datetime(d["start_time"]).dt.hour
    d["hour_bucket"] = pd.cut(d["hour"], bins=[-1,5,11,17,23], labels=["00-05", "06-11", "12-17", "18-23"])
    for method, fname in [("MAR (LDS)", "heatmap_mar_len_hour_rmse.png"), ("MNAR (LDS)", "heatmap_mnar_len_hour_rmse.png")]:
        dm = d[d["method"]==method]
        if dm.empty: continue
        rows = []
        for (lb, hb), g in dm.groupby(["length_bucket", "hour_bucket"], observed=True):
            rows.append({"length_bucket": str(lb), "hour_bucket": str(hb), "rmse": rmse_from_sqerr(g)})
        hm = pd.DataFrame(rows)
        piv = hm.pivot(index="length_bucket", columns="hour_bucket", values="rmse")
        row_order = [b for b in LENGTH_LABELS if b in piv.index]
        col_order = [c for c in ["00-05", "06-11", "12-17", "18-23"] if c in piv.columns]
        piv = piv.loc[row_order, col_order]
        plt.figure(figsize=(6.2, 4.2))
        plt.imshow(piv.to_numpy(dtype=float), aspect="auto")
        plt.colorbar(label="RMSE (mph)")
        plt.xticks(np.arange(len(col_order)), col_order)
        plt.yticks(np.arange(len(row_order)), row_order)
        plt.xlabel("Hour-of-day bucket")
        plt.ylabel("Blackout length bucket")
        plt.title(f"Seattle imputation RMSE: {method}")
        savefig(FIG_DIR / fname)
else:
    print("Need blackout_len and start_time for heatmaps; skipping.")


In [ ]:
# ============================================================
# 7) EM objective plot, synthetic alpha sweep, and METR-LA table
# ============================================================
# EM logs: auto-detect files with iteration/objective/loglikelihood columns
em_frames = []
for f in candidate_csv_files():
    try:
        df = pd.read_csv(f)
    except Exception:
        continue
    iter_col = first_col(df, ["iteration", "iter", "em_iter"])
    obj_col = first_col(df, ["objective", "loglik", "log_likelihood", "loglikelihood", "ll"])
    method_col = first_col(df, ["method", "model", "model_name"])
    if iter_col and obj_col:
        tmp = pd.DataFrame({
            "iteration": pd.to_numeric(df[iter_col], errors="coerce"),
            "objective": pd.to_numeric(df[obj_col], errors="coerce"),
            "method": df[method_col].map(clean_method) if method_col else clean_method(f.stem),
            "source_file": str(f.relative_to(ROOT))
        }).dropna(subset=["iteration", "objective"])
        if not tmp.empty:
            em_frames.append(tmp)

if em_frames:
    em = pd.concat(em_frames, ignore_index=True).drop_duplicates()
    em.to_csv(TABLE_DIR / "em_objective_logs_combined.csv", index=False)
    plt.figure(figsize=(7.2, 4.2))
    for method in method_order_list(em["method"].unique()):
        gm = em[em["method"]==method].sort_values("iteration")
        if gm.empty: continue
        plt.plot(gm["iteration"], gm["objective"], marker="o", label=method)
    plt.xlabel("EM iteration")
    plt.ylabel("Training objective")
    plt.title("Approximate-EM training objective")
    plt.legend(frameon=False)
    savefig(FIG_DIR / "em-training.png")
else:
    print("No EM objective CSV detected. If reviewer asks, use existing em-training.png or save logs with columns iteration, method, objective.")

# Synthetic alpha sweep summary from pred if alpha present, or from aggregate CSVs
alpha_pred = pred[pred["alpha"].notna()].copy()
if not alpha_pred.empty:
    alpha_rows = []
    for (alpha, method, seed), g in alpha_pred.groupby(["alpha", "method", "seed"], dropna=False):
        alpha_rows.append({"alpha": alpha, "method": method, "seed": seed, "rmse": rmse_from_sqerr(g)})
    alpha_long = pd.DataFrame(alpha_rows)
    alpha_long.to_csv(TABLE_DIR / "synthetic_alpha_seed_rmse_long.csv", index=False)
    alpha_summary = alpha_long.groupby(["alpha", "method"], as_index=False).agg(mean_rmse=("rmse", "mean"), std_rmse=("rmse", "std"), n_seeds=("seed", "nunique"))
    alpha_summary.to_csv(TABLE_DIR / "synthetic_alpha_summary_by_method.csv", index=False)
    # Delta table
    piv = alpha_long.pivot_table(index=["alpha", "seed"], columns="method", values="rmse", aggfunc="first").reset_index()
    if "MNAR (LDS)" in piv.columns and "MAR (LDS)" in piv.columns:
        piv["delta_rmse"] = piv["MNAR (LDS)"] - piv["MAR (LDS)"]
        delta = piv.groupby("alpha", as_index=False).agg(delta_mean=("delta_rmse", "mean"), delta_std=("delta_rmse", "std"), n_seeds=("seed", "nunique"))
        delta.to_csv(TABLE_DIR / "synthetic_alpha_sweep_delta.csv", index=False)
        plt.figure(figsize=(6.4, 4.0))
        x = delta["alpha"].to_numpy(dtype=float)
        y = delta["delta_mean"].to_numpy(dtype=float)
        yerr = delta["delta_std"].fillna(0).to_numpy(dtype=float)
        plt.axhline(0, linestyle="--", linewidth=1)
        plt.errorbar(x, y, yerr=yerr, marker="o", capsize=4)
        plt.xlabel(r"State-dependence strength $\alpha$")
        plt.ylabel(r"$\Delta$RMSE = MNAR - MAR")
        plt.title("Synthetic MNAR-strength sweep")
        savefig(FIG_DIR / "synthetic_validation_alpha_sweep.png")
        print(delta.round(4))
else:
    print("No alpha column found. If synthetic sweep is in a separate CSV, make sure it has columns alpha, method, seed, rmse or y_true/y_pred.")

# METR-LA wide table
metr = main[main["dataset"]=="METR-LA"].copy()
if not metr.empty:
    piv = metr.pivot_table(index="method", columns="task_horizon", values="rmse", aggfunc="first")
    cols = [c for c in HORIZON_ORDER if c in piv.columns] + [c for c in piv.columns if c not in HORIZON_ORDER]
    idx = [m for m in PAPER_METHOD_ORDER if m in piv.index] + [m for m in piv.index if m not in PAPER_METHOD_ORDER]
    piv = piv.loc[idx, cols]
    piv.to_csv(TABLE_DIR / "metrla_rmse_wide.csv")
    print("METR-LA table saved:", TABLE_DIR / "metrla_rmse_wide.csv")
    print(piv.round(3))


In [ ]:
# ============================================================
# 8) Optional missingness diagnostics collector
# ============================================================
# This cell tries to collect diagnostic CSVs if notebooks 1--6 saved them.
# Expected names/columns are flexible: auc, roc_auc, diagnostic/test/model/features.
diag_rows = []
for f in candidate_csv_files():
    lowname = f.name.lower()
    if not any(k in lowname for k in ["diagnostic", "diag", "auc", "missingness", "blackout"]):
        continue
    try:
        df = pd.read_csv(f)
    except Exception:
        continue
    auc_col = first_col(df, ["auc", "roc_auc", "auroc"])
    if auc_col is None:
        continue
    test_col = first_col(df, ["test", "diagnostic", "experiment", "name"])
    feat_col = first_col(df, ["features", "feature_set", "model", "classifier"])
    for _, r in df.iterrows():
        diag_rows.append({
            "source_file": str(f.relative_to(ROOT)),
            "diagnostic": str(r[test_col]) if test_col else f.stem,
            "feature_set": str(r[feat_col]) if feat_col else "unspecified",
            "auc": pd.to_numeric(r[auc_col], errors="coerce"),
        })

diag = pd.DataFrame(diag_rows).dropna(subset=["auc"]) if diag_rows else pd.DataFrame()
if not diag.empty:
    diag.to_csv(TABLE_DIR / "missingness_diagnostics_auc.csv", index=False)
    print("Saved:", TABLE_DIR / "missingness_diagnostics_auc.csv")
    print(diag.round(4))
else:
    print("No diagnostic AUC CSVs detected.")
    print("To satisfy the reviewer cleanly, save a CSV like missingness_diagnostics.csv with columns:")
    print("diagnostic, feature_set, auc")
    print("Example rows:")
    print("onset_vs_control, observed_edges, 0.533")
    print("next_step_missingness, observed_only, 0.518")
    print("next_step_missingness, latent_augmented, 0.661")


In [ ]:
# ============================================================
# 9) Final paper-number summary and sanity checks
# ============================================================
summary_lines = []
summary_lines.append("# Final Reviewer Run Summary\n")
summary_lines.append(f"Generated outputs folder: `{OUT}`\n")

# Main tables
summary_lines.append("## Main RMSE tables\n")
for dataset in main["dataset"].unique():
    piv = main[main["dataset"]==dataset].pivot_table(index="method", columns="task_horizon", values="rmse", aggfunc="first")
    cols = [c for c in HORIZON_ORDER if c in piv.columns] + [c for c in piv.columns if c not in HORIZON_ORDER]
    idx = [m for m in PAPER_METHOD_ORDER if m in piv.index] + [m for m in piv.index if m not in PAPER_METHOD_ORDER]
    piv = piv.loc[idx, cols]
    summary_lines.append(f"### {dataset}\n")
    summary_lines.append(piv.round(3).to_markdown())
    summary_lines.append("\n")

if 'boot' in globals() and isinstance(boot, pd.DataFrame) and not boot.empty:
    summary_lines.append("## Paired bootstrap MNAR - MAR deltas\n")
    b2 = boot.drop(columns=[c for c in ["horizon_order"] if c in boot.columns]).copy()
    summary_lines.append(b2.round(4).to_markdown(index=False))
    summary_lines.append("\n")

# Reviewer checklist
summary_lines.append("## Reviewer-request checklist\n")
checklist = {
    "Main aligned-window RMSE table": TABLE_DIR / "main_rmse_long.csv",
    "Seattle imputation CIs figure": FIG_DIR / "impute_rmse_methods_ci_labels.png",
    "Forecast CIs figure": FIG_DIR / "forecast_rmse_by_horizon_panels_ci.png",
    "Paired MNAR-MAR bootstrap deltas": TABLE_DIR / "paired_bootstrap_mnar_minus_mar.csv",
    "Length-bucket plot with empty buckets dropped": FIG_DIR / "impute_rmse_by_length_bucket_counts.png",
    "EM objective diagnostic": FIG_DIR / "em-training.png",
    "Seed robustness summary if seeds were rerun": TABLE_DIR / "seed_robustness_summary.csv",
    "Missingness diagnostics if saved by earlier notebooks": TABLE_DIR / "missingness_diagnostics_auc.csv",
    "Synthetic alpha sweep delta table": TABLE_DIR / "synthetic_alpha_sweep_delta.csv",
    "Synthetic alpha sweep figure": FIG_DIR / "synthetic_validation_alpha_sweep.png",
}
for name, path in checklist.items():
    ok = path.exists()
    summary_lines.append(f"- [{'x' if ok else ' '}] {name}: `{path}`")
summary_lines.append("\n")

summary_path = OUT / "paper_numbers_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))
print("\nSaved summary:", summary_path)

# LaTeX helper for bootstrap table if available
if 'boot' in globals() and isinstance(boot, pd.DataFrame) and not boot.empty:
    latex_boot = boot.copy()
    latex_boot = latex_boot[latex_boot["dataset"].eq("Seattle")]
    if not latex_boot.empty:
        latex_boot["Delta RMSE"] = latex_boot["delta_rmse"].map(lambda x: f"{x:.3f}")
        latex_boot["95% CI"] = latex_boot.apply(lambda r: f"[{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)
        latex_small = latex_boot[["task_horizon", "Delta RMSE", "95% CI", "n_windows"]].rename(columns={"task_horizon":"Task", "n_windows":"Windows"})
        (TABLE_DIR / "latex_bootstrap_table_preview.txt").write_text(latex_small.to_latex(index=False, escape=False), encoding="utf-8")
        print("Saved LaTeX preview:", TABLE_DIR / "latex_bootstrap_table_preview.txt")


## What to do after this notebook runs

Use the generated `final_reviewer_outputs/paper_numbers_summary.md` as the source of truth for the revision. In the paper, only claim seed robustness if `seed_robustness_summary.csv` exists and contains multiple seeds. If the notebook only produced bootstrap CIs, phrase it as window-level uncertainty instead.
